# Step 06 - End-to-End Demo

Goal: run full pipeline (research -> writer -> editor -> publisher) and publish to Dev.to.

## What's New in This Step

- Step 05 produced final article JSON locally.
- This step adds a custom publishing tool and a publisher agent.
- Pipeline depth increases to research -> writer -> editor -> publisher with draft-first safety.

### Setup

In [ ]:
import json
import os
import requests
from dotenv import load_dotenv
from crewai import LLM, Agent, Crew, Task
from crewai.tools import BaseTool
from crewai_tools import SerperDevTool


load_dotenv()

# Silence the telemetry HTTPS read-timeout that otherwise prints during a demo.
os.environ.setdefault("CREWAI_TELEMETRY_OPT_OUT", "1")

# The pipeline is topic-agnostic — change TOPIC to anything to see the same
# research -> write -> edit -> publish flow run end-to-end.
TOPIC = "Is DevOps/Platform engineering relevant in AI Era"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
serper_api_key = os.getenv("SERPER_API_KEY")
devto_api_key = os.getenv("DEVTO_API_KEY")

if not openrouter_api_key:
    raise ValueError("Missing OPENROUTER_API_KEY")

if not serper_api_key:
    raise ValueError("Missing SERPER_API_KEY")

if not devto_api_key:
    raise ValueError("Missing DEVTO_API_KEY")

### Helper function

In [ ]:
def extract_json_object(text):
    # Format guardrail recovery: parse the first valid JSON object in model output.
    cleaned = (
        text.strip()
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

    decoder = json.JSONDecoder()

    for i, char in enumerate(cleaned):
        if char != "{":
            continue

        try:
            obj, _ = decoder.raw_decode(cleaned[i:])
            return obj
        except json.JSONDecodeError:
            continue

    raise ValueError(
        "Could not extract valid JSON object from model output."
    )

### Custom tool for publishing to Dev.to via their API.

In [ ]:
class DevtoPublishTool(BaseTool):
    # Custom tool class: wraps the Dev.to API for agent-driven publishing.
    # The api key is read from the environment inside _run rather than
    # captured by closure, so this class also works when copied into a .py
    # file with no module-level globals available.
    name: str = "devto_publish"
    description: str = "Publish an article JSON string to Dev.to"

    def _run(self, article_json: str) -> str:
        api_key = os.getenv("DEVTO_API_KEY")
        if not api_key:
            raise RuntimeError("DEVTO_API_KEY is not set")

        article = json.loads(article_json)

        payload = {
            "article": {
                "title": article["title"],
                "body_markdown": article["content"],
                "tags": article["tags"],
                "published": False,
            }
        }

        response = requests.post(
            "https://dev.to/api/articles",
            json=payload,
            headers={
                "api-key": api_key,
                "Content-Type": "application/json",
            },
            timeout=30,
        )

        response.raise_for_status()
        return response.text

### Create LLM and Agents

In [ ]:
# LLM: shared model used by all agents in this pipeline.
llm = LLM(
    model="openai/gpt-4o",
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,
)

# Built-in search tool for fresh references.
search_tool = SerperDevTool()
# Custom publish tool to for dev.to
publish_tool = DevtoPublishTool()

# Agent 1: discovers fresh source-backed information.
research_agent = Agent(
    role="Research Analyst",
    goal="Find latest updates on {topic}",
    backstory="You give short, factual, source-backed summaries.",
    llm=llm,
    tools=[search_tool],
    verbose=False,
)

# Agent 2: turns research findings into markdown.
writer_agent = Agent(
    role="Content Writer",
    goal="Write a simple blog post on {topic}",
    backstory="You write clear and beginner-friendly content.",
    llm=llm,
    verbose=False,
)

# Agent 3: enforces output guardrails and returns strict JSON.
editor_agent = Agent(
    role="Editor",
    goal="Return final output in JSON",
    backstory="You return only valid JSON with title, tags, and content.",
    llm=llm,
    verbose=False,
)

# Agent 4: executes publishing and returns exact API response.
publisher_agent = Agent(
    role="Publisher Agent",
    goal="Publish the final article JSON to Dev.to",
    backstory="You publish approved content and return the exact API response.",
    llm=llm,
    tools=[publish_tool],
    verbose=False,
)


### Define Tasks with Context

In [ ]:
# Task 1: collect recent source-backed notes.
research_task = Task(
    description="Research {topic} from 2026 onward with source links.",
    expected_output="A short research summary with sources.",
    agent=research_agent,
)

# Task 2: generate Markdown from research context.
writing_task = Task(
    description="Write a clear blog post from the research notes.",
    expected_output="A markdown blog post.",
    agent=writer_agent,
    context=[research_task],
)

# Task 3: convert Markdown to final article JSON.
# - Format guardrail: valid JSON only
# - Shape guardrail: required keys title/tags/content
# - Quality guardrail: up to 4 short tags
editing_task = Task(
    description=(
        "Return only valid JSON with keys: title, tags, content. "
        "tags must be a list of up to 4 short tags."
    ),
    expected_output='{"title": "...", "tags": ["ai"], "content": "markdown"}',
    agent=editor_agent,
    context=[writing_task],
)

### Create Content Crew with Agents and Tasks to Generate Final Article JSON

In [ ]:
content_crew = Crew(
    agents=[research_agent, writer_agent, editor_agent],
    tasks=[research_task, writing_task, editing_task],
    verbose=False,
)

content_result = content_crew.kickoff(inputs={"topic": TOPIC})
content_text = getattr(content_result, "raw", str(content_result))

article_data = extract_json_object(content_text)

# Quality guardrail normalization before publish.
raw_tags = article_data.get("tags", [])

if isinstance(raw_tags, str):
    raw_tags = [raw_tags]

article_data["tags"] = [
    str(tag).lower().replace(" ", "")
    for tag in raw_tags
][:4]

if not article_data["tags"]:
    article_data["tags"] = ["ai", "agents"]

article_json = json.dumps(article_data, ensure_ascii=False)

print(json.dumps(article_data, indent=2))

### Create publish task and Publish crew

In [ ]:
# Task 4: publish normalized JSON through the custom tool.
# Note on braces: Crew renders `{article_json}` from kickoff(inputs=...).
# Any literal `{` / `}` in description or expected_output is also interpreted
# as a template placeholder, so JSON examples here must either be escaped
# (`{{`, `}}`) or — as we do below — kept out of the prompt entirely.
publish_task = Task(
    description=(
        "Use the devto_publish tool exactly once with this article JSON:\n"
        "{article_json}"
    ),
    expected_output="The exact API response returned after publishing.",
    agent=publisher_agent,
)

publish_crew = Crew(
    agents=[publisher_agent],
    tasks=[publish_task],
    verbose=False,
)

publish_result = publish_crew.kickoff(
    inputs={"article_json": article_json}
)

print(getattr(publish_result, "raw", str(publish_result)))

In [ ]:
# Cost / latency visibility: GPT-4o across 4 agents + tool calls is not free.
# CrewAI exposes per-run usage on the Crew object after kickoff.
print("Content crew usage:", content_crew.usage_metrics)
print("Publish crew usage:", publish_crew.usage_metrics)

### Recap
- LLM did: power reasoning and text generation across all stages.
- Agents did: split responsibilities into research, writing, editing, and publishing.
- Task enforced: staged contracts, guardrails, and safe draft-first publish behavior.